In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/processed_data.csv')

X = df.drop(columns=['DESYNPUF_ID', 'total_cost_2010'])
y = df['total_cost_2010']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)
print(X_train.dtypes)

(74846, 19) (18712, 19)
age                    int64
BENE_SEX_IDENT_CD      int64
BENE_RACE_CD           int64
BENE_ESRD_IND          int64
SP_ALZHDMTA            int64
SP_CHF                 int64
SP_CHRNKIDN            int64
SP_CNCR                int64
SP_COPD                int64
SP_DEPRESSN            int64
SP_DIABETES            int64
SP_ISCHMCHT            int64
SP_OSTEOPRS            int64
SP_RA_OA               int64
SP_STRKETIA            int64
chronic_count          int64
cost_2008            float64
cost_2009            float64
cost_trend           float64
dtype: object


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler

# baseline
train_df = X_train.copy()
train_df['total_cost_2010'] = y_train.values
baseline_lookup = train_df.groupby('chronic_count')['total_cost_2010'].mean()
baseline_preds = X_test['chronic_count'].map(baseline_lookup)
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))
print(f"Baseline - MAE: {baseline_mae:.2f}, RMSE: {baseline_rmse:.2f}")

# linear/ridge
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train_log)
lin_preds = np.expm1(lin_model.predict(X_test_scaled))

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train_log)
ridge_preds = np.expm1(ridge_model.predict(X_test_scaled))

for name, preds in [('Linear', lin_preds), ('Ridge', ridge_preds)]:
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{name} - MAE: {mae:.2f}, RMSE: {rmse:.2f}")

Baseline - MAE: 2695.69, RMSE: 5603.56
Linear - MAE: 2491.35, RMSE: 5976.78
Ridge - MAE: 2491.34, RMSE: 5976.78


In [17]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # note: raw y, not log, and unscaled X
rf_preds = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
print(f"Random Forest - MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}")

Random Forest - MAE: 2571.19, RMSE: 5524.42


In [18]:
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)

cost_2009            0.350716
cost_2008            0.262671
cost_trend           0.102841
age                  0.087790
chronic_count        0.048005
BENE_ESRD_IND        0.027501
BENE_RACE_CD         0.019797
SP_CHRNKIDN          0.012411
BENE_SEX_IDENT_CD    0.011143
SP_COPD              0.009842
SP_ALZHDMTA          0.008721
SP_RA_OA             0.008670
SP_DEPRESSN          0.008592
SP_OSTEOPRS          0.008131
SP_DIABETES          0.007996
SP_ISCHMCHT          0.006837
SP_CHF               0.006676
SP_CNCR              0.006074
SP_STRKETIA          0.005586
dtype: float64


In [19]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
print(f"XGBoost - MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}")

XGBoost - MAE: 2567.49, RMSE: 5511.41


In [20]:
xgb_train_preds = xgb_model.predict(X_train)
train_mae = mean_absolute_error(y_train, xgb_train_preds)
train_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_preds))
print(f"XGBoost (train) - MAE: {train_mae:.2f}, RMSE: {train_rmse:.2f}")

XGBoost (train) - MAE: 2485.35, RMSE: 5402.33


In [21]:
test_results = pd.DataFrame({
    'DESYNPUF_ID': df.loc[X_test.index, 'DESYNPUF_ID'].values,
    'actual_cost': y_test.values,
    'predicted_cost': xgb_preds
})

low_cutoff = test_results['predicted_cost'].quantile(0.50)
high_cutoff = test_results['predicted_cost'].quantile(0.85)

def assign_tier(cost):
    if cost <= low_cutoff:
        return 'Low'
    elif cost <= high_cutoff:
        return 'Medium'
    else:
        return 'High'

test_results['risk_tier'] = test_results['predicted_cost'].apply(assign_tier)

print(f"Low cutoff: ${low_cutoff:.2f}")
print(f"High cutoff: ${high_cutoff:.2f}")
print(test_results['risk_tier'].value_counts())
print(test_results.groupby('risk_tier')['actual_cost'].mean())

Low cutoff: $2824.44
High cutoff: $4449.72
risk_tier
Low       9356
Medium    6549
High      2807
Name: count, dtype: int64
risk_tier
High      5533.323833
Low       1294.445276
Medium    3688.862422
Name: actual_cost, dtype: float64


In [22]:
total_cost = test_results['actual_cost'].sum()
high_tier_cost = test_results[test_results['risk_tier'] == 'High']['actual_cost'].sum()
print(f"High tier: {len(test_results[test_results['risk_tier']=='High'])/len(test_results)*100:.1f}% of patients, "
      f"{high_tier_cost/total_cost*100:.1f}% of total cost")

High tier: 15.0% of patients, 30.0% of total cost


In [23]:
import joblib

joblib.dump(xgb_model, '../src/xgb_cost_model.pkl')
joblib.dump({'low_cutoff': low_cutoff, 'high_cutoff': high_cutoff}, '../src/risk_tier_cutoffs.pkl')

print("Saved.")

Saved.


In [29]:
test_results.to_csv('../data/processed/test_predictions.csv', index=False)